# Embedding Generation 

The current pipeline:
- **Milvus Lite** as the vector database
- **BAAI/bge-m3** as the embedding model (via Hugging Face transformers, dense CLS embeddings)
- **schema** from `cco schema.json`
- **ODP patterns** from `Ontology design patterns.json`
- **Final units** from `preparation_v1`

In [ ]:
import sys
!{sys.executable} -m pip install -q --upgrade "transformers>=4.41,<5.0.0" sentencepiece accelerate "pymilvus[milvus_lite]"
print("Installed packages for BGE-M3 via Hugging Face transformers and Milvus Lite.")
print("Pinned transformers to <5.0.0 to stay compatible with sentence-transformers already present in the environment.")


In [ ]:
import os

# ── Paths ────────────────────────────────────────────────────────────────────
SCHEMA_JSON = "../ODPs & CCO Schema/Schema Extraction/output/cco schema_v1.json"
ODP_JSON    = "../ODPs & CCO Schema/ODPs/ODP.json"

Final_Chunks_Version = "v1"
Final_Chunks_Dir = f"../PDFs Processing/output/preparation_{Final_Chunks_Version}"
Final_Chunks_Files = {
    "UK":        f"final_Chunks_uk_{Final_Chunks_Version}.json",
    "Canada":    f"final_Chunks_canada_{Final_Chunks_Version}.json",
    "Australia": f"final_Chunks_australia_{Final_Chunks_Version}.json",
    "USA":       f"final_Chunks_usa_{Final_Chunks_Version}.json",
}

OUTPUT_DIR      = "output"
MILVUS_URI      = os.path.join(OUTPUT_DIR, "milvus_bge_m3.db")  # Milvus Lite local file
EMBEDDING_MODEL = "BAAI/bge-m3"
USE_FP16        = False
BATCH_SIZE      = 32
MAX_LENGTH      = 1024

SCHEMA_COLLECTION = "cco_Schema"
ODP_COLLECTION    = "ODP"
UNITS_COLLECTION  = f"final_chunks{Final_Chunks_Version}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Schema JSON      : {SCHEMA_JSON}  exists={os.path.exists(SCHEMA_JSON)}")
print(f"  ODP JSON         : {ODP_JSON}  exists={os.path.exists(ODP_JSON)}")
print(f"  Prepared units   : {Final_Chunks_Dir}")
print(f"  Milvus URI       : {MILVUS_URI}")
print(f"  Embedding model  : {EMBEDDING_MODEL}")
for jur, fname in Final_Chunks_Files.items():
    path = os.path.join(Final_Chunks_Dir, fname)
    print(f"  {jur}: exists={os.path.exists(path)}")


In [ ]:
import json
import time
from datetime import datetime
from typing import Any, Dict, List

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from pymilvus import (
    connections,
    utility,
    FieldSchema,
    CollectionSchema,
    DataType,
    Collection,
)
import warnings

print("Imports OK")


In [ ]:
import transformers, pymilvus, torch
print("transformers:", transformers.__version__)
print("pymilvus:", pymilvus.__version__)
print("torch:", torch.__version__)


In [ ]:
# ── Load embedding model ─────────────────────────────────────────────────────
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading model: {EMBEDDING_MODEL} on {DEVICE} ...")
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
model = AutoModel.from_pretrained(EMBEDDING_MODEL)
model.to(DEVICE)
model.eval()

# BGE-M3 dense embedding uses the normalized hidden state of [CLS]
with torch.no_grad():
    probe_inputs = tokenizer(["probe text"], padding=True, truncation=True, max_length=16, return_tensors="pt")
    probe_inputs = {k: v.to(DEVICE) for k, v in probe_inputs.items()}
    probe_outputs = model(**probe_inputs)
    probe_vec = F.normalize(probe_outputs.last_hidden_state[:, 0], p=2, dim=1)
    DENSE_DIM = probe_vec.shape[1]
print(f"  Model loaded. Dense dim: {DENSE_DIM}")

# ── Initialize Milvus Lite ───────────────────────────────────────────────────
connections.connect(alias="default", uri=MILVUS_URI)
print(f"Milvus Lite initialized at: {MILVUS_URI}")
print(f"Existing collections: {utility.list_collections()}")


In [ ]:
def embed_dense(texts: List[str], batch_size: int = BATCH_SIZE) -> List[List[float]]:
    """Generate dense BGE-M3 embeddings using normalized [CLS] states."""
    all_vecs: List[List[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            vecs = F.normalize(outputs.last_hidden_state[:, 0], p=2, dim=1)
        all_vecs.extend(vecs.cpu().tolist())
    return all_vecs


def ensure_milvus_connection() -> None:
    """Reconnect defensively so notebook cells work after kernel restarts or out-of-order execution."""
    connections.connect(alias="default", uri=MILVUS_URI)


def serialize_value(value: Any) -> Any:
    if value is None:
        return ""
    if isinstance(value, (str, int, float, bool)):
        return value
    return json.dumps(value, ensure_ascii=False)


def recreate_collection(name: str, description: str) -> Collection:
    ensure_milvus_connection()
    if utility.has_collection(name):
        print(f"  Collection '{name}' exists — recreating fresh.")
        utility.drop_collection(name)

    fields = [
        FieldSchema(name="pk", dtype=DataType.VARCHAR, is_primary=True, auto_id=False, max_length=128),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
        FieldSchema(name="dense_vector", dtype=DataType.FLOAT_VECTOR, dim=DENSE_DIM),
    ]
    schema = CollectionSchema(fields=fields, description=description, enable_dynamic_field=True)
    collection = Collection(name=name, schema=schema, consistency_level="Strong")
    collection.create_index(
        field_name="dense_vector",
        index_params={"index_type": "AUTOINDEX", "metric_type": "COSINE", "params": {}},
    )
    collection.load()
    print(f"  Collection '{name}' created.")
    return collection


def insert_rows(collection: Collection, rows: List[Dict[str, Any]], batch_size: int = 128) -> int:
    total = 0
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        collection.insert(batch)
        total += len(batch)
    collection.flush()
    return total


def build_unit_embedding_text(unit: Dict[str, Any]) -> str:
    section_path = " > ".join(unit.get("section_path") or [])
    local_heading = (unit.get("local_heading") or "").strip()
    parts = [
        f"Jurisdiction: {unit.get('jurisdiction', '')}.",
        f"Unit type: {unit.get('unit_type', '')}.",
    ]
    if section_path:
        parts.append(f"Section path: {section_path}.")
    if local_heading:
        parts.append(f"Local heading: {local_heading}.")
    parts.append(f"Text: {(unit.get('text') or '').strip()}")
    return " ".join(parts).strip()


print("Helper functions defined.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EMBED CCO SCHEMA (final)
# retrieval_text only
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("EMBEDDING: CCO Schema (final)")
print("=" * 70)

with open(SCHEMA_JSON, encoding="utf-8") as f:
    schema_data = json.load(f)

schema_items = schema_data["classes"] + schema_data["object_properties"] + schema_data["data_properties"]
print(f"Total schema items: {len(schema_items)}")

collection_schema = recreate_collection(
    SCHEMA_COLLECTION,
    "CCO schema embeddings for Layer 2 schema-grounded candidate gating",
)

texts = []
base_rows = []
for item in schema_items:
    text = (item.get("retrieval_text") or "").strip()
    if not text:
        print(f"  WARNING: no retrieval_text for {item.get('id', '?')} — skipping")
        continue
    texts.append(text)
    base_rows.append({
        "pk": item["id"],
        "text": text,
        "item_id": item["id"],
        "item_type": item["item_type"],
        "collection_name": "cco_schema",
        "label": item.get("label", item.get("local_name", "")),
        "local_name": item.get("local_name", ""),
        "schema_group": item.get("collection", "cco_schema"),
        "source_uri": item.get("uri", ""),
    })

print(f"Generating dense embeddings for {len(texts)} schema items...")
t0 = time.time()
dense_vectors = embed_dense(texts)
print(f"  Done in {time.time() - t0:.1f}s")

rows = []
for row, vec in zip(base_rows, dense_vectors):
    item = dict(row)
    item["dense_vector"] = vec
    rows.append(item)

stored = insert_rows(collection_schema, rows)
print(f"Stored {stored} vectors in '{SCHEMA_COLLECTION}'")
print(f"Collection count: {collection_schema.num_entities}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EMBED ODP LIBRARY LAYER 3
# retrieval_text only
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("EMBEDDING: ODP Library Layer 3")
print("=" * 70)

with open(ODP_JSON, encoding="utf-8") as f:
    odp_data = json.load(f)

patterns = []
for cat_key, cat in odp_data["categories"].items():
    for pattern in cat.get("patterns", []):
        if pattern.get("pipeline_layer") != "Layer 3":
            continue
        p = dict(pattern)
        p["category_key"] = cat_key
        patterns.append(p)

print(f"Total Layer 3 patterns: {len(patterns)}")

collection_odp = recreate_collection(
    ODP_COLLECTION,
    "ODP Layer 3 retrieval embeddings",
)

texts = []
base_rows = []
for pattern in patterns:
    text = (pattern.get("retrieval_text") or "").strip()
    if not text:
        print(f"  WARNING: no retrieval_text for {pattern.get('id', '?')} — skipping")
        continue
    texts.append(text)
    base_rows.append({
        "pk": pattern["id"],
        "text": text,
        "item_id": pattern["id"],
        "item_type": "odp_pattern",
        "collection_name": "odp_library",
        "label": pattern.get("label", ""),
        "category": pattern.get("category", ""),
        "category_key": pattern.get("category_key", ""),
        "pipeline_layer": pattern.get("pipeline_layer", ""),
    })

print(f"Generating dense embeddings for {len(texts)} ODP patterns...")
t0 = time.time()
dense_vectors = embed_dense(texts)
print(f"  Done in {time.time() - t0:.1f}s")

rows = []
for row, vec in zip(base_rows, dense_vectors):
    item = dict(row)
    item["dense_vector"] = vec
    rows.append(item)

stored = insert_rows(collection_odp, rows)
print(f"Stored {stored} vectors in '{ODP_COLLECTION}'")
print(f"Collection count: {collection_odp.num_entities}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EMBED PREPARED UNITS v19_6
# uses Layer 1 prepared units, not old v15 provisions
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("EMBEDDING: Final chunks")
print("=" * 70)

collection_units = recreate_collection(
    UNITS_COLLECTION,
    "Final chunks embeddings for Layer 2 schema-grounded candidate gating",
)

all_rows = []
for jurisdiction, filename in Final_Chunks_Files.items():
    path = os.path.join(Final_Chunks_Dir, filename)
    if not os.path.exists(path):
        print(f"  WARNING: {path} not found — skipping")
        continue

    with open(path, encoding="utf-8") as f:
        payload = json.load(f)
    units = payload["final_chunks"]
    print()
    print(f"{jurisdiction}: {len(units)} prepared units")

    texts = []
    base_rows = []
    for unit in units:
        raw_text = (unit.get("text") or "").strip()
        if not raw_text:
            print(f"  WARNING: empty text for {unit.get('unit_id', '?')} — skipping")
            continue

        embedding_text = build_unit_embedding_text(unit)
        texts.append(embedding_text)
        base_rows.append({
            "pk": unit["unit_id"],
            "text": embedding_text,
            "raw_text": raw_text,
            "unit_id": unit["unit_id"],
            "item_type": "prepared_unit",
            "collection_name": "final_chunks",
            "jurisdiction": unit.get("jurisdiction", jurisdiction),
            "source_file": unit.get("source_file", ""),
            "unit_type": unit.get("unit_type", ""),
            "pdf_page_start": int(unit.get("pdf_page_start") or 0),
            "pdf_page_end": int(unit.get("pdf_page_end") or 0),
            "printed_page_start": serialize_value(unit.get("printed_page_start")),
            "printed_page_end": serialize_value(unit.get("printed_page_end")),
            "section": serialize_value(unit.get("section")),
            "section_id": serialize_value(unit.get("section_id")),
            "section_path": " > ".join(unit.get("section_path") or []),
            "local_heading": serialize_value(unit.get("local_heading")),
            "paragraph_number": serialize_value(unit.get("paragraph_number")),
            "source_anchor": serialize_value(unit.get("source_anchor")),
            "component_chunk_ids": serialize_value(unit.get("component_chunk_ids")),
            "support_chunk_ids": serialize_value(unit.get("support_chunk_ids")),
        })

    print(f"  Generating dense embeddings for {len(texts)} units...")
    t0 = time.time()
    dense_vectors = embed_dense(texts)
    print(f"  Done in {time.time() - t0:.1f}s")

    rows = []
    for row, vec in zip(base_rows, dense_vectors):
        item = dict(row)
        item["dense_vector"] = vec
        rows.append(item)

    stored = insert_rows(collection_units, rows)
    all_rows.extend(rows)
    print(f"  Stored: {stored} unit vectors")

print(f"Collection count: {collection_units.num_entities}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VERIFICATION — SEARCH EACH COLLECTION
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("VERIFICATION — Milvus Collections")
print("=" * 70)

collections = {
    SCHEMA_COLLECTION: collection_schema,
    ODP_COLLECTION: collection_odp,
    UNITS_COLLECTION: collection_units,
}

query = "provider must submit evidence of eligibility from 1 August 2025"
query_vec = embed_dense([query], batch_size=1)[0]

for name, col in collections.items():
    print()
    print(f"{name}: {col.num_entities} vectors")
    results = col.search(
        data=[query_vec],
        anns_field="dense_vector",
        param={"metric_type": "COSINE", "params": {}},
        limit=3,
        output_fields=["text", "label", "item_id", "unit_id", "jurisdiction", "unit_type", "section", "source_anchor"],
    )
    print(f"  Query: {query}")
    for i, hit in enumerate(results[0], start=1):
        entity = hit.get("entity", {})
        label = entity.get("label") or entity.get("unit_id") or entity.get("item_id") or "?"
        print(f"    {i}. [{label}] score={hit.get('distance', 0):.4f}")
        print(f"       {entity.get('text', '')[:160]}...")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE EMBEDDING LOG
# ══════════════════════════════════════════════════════════════════════════════
log = {
    "created_at": datetime.now().isoformat(),
    "embedding_model": EMBEDDING_MODEL,
    "milvus_uri": MILVUS_URI,
    "design_note": "Milvus Lite + BGE-M3 dense embeddings. Schema and ODP use retrieval_text; final chunks use a structured embedding text built from chunk metadata and text.",
    "collections": {
        SCHEMA_COLLECTION: {
            "count": collection_schema.num_entities,
            "description": "CCO schema classes + object properties + data properties",
            "fields": ["retrieval_text"],
        },
        ODP_COLLECTION: {
            "count": collection_odp.num_entities,
            "description": "Ontology design patterns",
            "fields": ["retrieval_text"],
        },
        UNITS_COLLECTION: {
            "count": collection_units.num_entities,
            "description": "Final chunks from UK, Canada, Australia, USA",
            "fields": ["structured embedding text from chunk metadata + raw text"],
        },
    },
    "total_vectors": collection_schema.num_entities + collection_odp.num_entities + collection_units.num_entities,
}

log_path = os.path.join(OUTPUT_DIR, "embedding_log.json")
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False)

print(f"Embedding log saved: {log_path}")
print(json.dumps(log, indent=2, ensure_ascii=False))
